**test the impact of forcast submission format on forecast skill score and performance analysis**

convert binned ILI forecasts to quantile format 

score both binned and quantile forecast formats with logarithmic score and WIS, respectively.

done for the ensemble and all benchmark models

In [4]:
import pandas as pd
import numpy as np
import datetime
from epiweeks import Week
from collections import defaultdict
from pathlib import Path
import requests
import os
from scipy.special import expit, logit

import warnings
warnings.filterwarnings('ignore')

In [2]:
import matplotlib.dates as mdates
def set_date_axis_fmt(ax):
    # Set the locator
    locator = mdates.MonthLocator(interval=3)  # every month
    # Specify the format
    fmt = mdates.DateFormatter('%b %y')

    X = ax.xaxis
    X.set_major_locator(locator)
    # Specify formatter
    X.set_major_formatter(fmt)
 

In [5]:
def get_7_quantiles_interp(group, max_bin_width=0.05):
    quantiles = [0.025, 0.1  , 0.25 , 0.5  , 0.75 , 0.9  , 0.975]

    group = group.sort_values('bin_start_incl').reset_index(drop=True)
    bin_widths = group['bin_end_notincl'] - group['bin_start_incl']

    out = []
    for q in quantiles:
        idx_list = group.index[group['cum_val'] >= q]

        if len(idx_list) == 0:
            out.append(group.iloc[-1]['bin_end_notincl'])
            continue

        idx = idx_list[0]
        row = group.loc[idx]
        prev_cum = group.loc[idx - 1, 'cum_val'] if idx > 0 else 0.0
        bin_width = bin_widths.iloc[idx]

        if bin_width <= max_bin_width:
            if row['cum_val'] > prev_cum:
                frac = (q - prev_cum) / (row['cum_val'] - prev_cum)
                value = row['bin_start_incl'] + frac * bin_width
            else:
                value = row['bin_start_incl']
        else:
            value = _extrapolate_from_previous_bin(
                group, idx, q, bin_widths, max_bin_width
            )

        out.append(value)

    return pd.DataFrame({'quantile': quantiles, 'value': out})



def get_23_quantiles_interp(group, max_bin_width=0.05):
    quantiles = [0.01, 0.025] + list(np.arange(0.05, 0.96, 0.05)) + [0.975, 0.99]
    quantiles = [round(q, 3) for q in quantiles]

    group = group.sort_values('bin_start_incl').reset_index(drop=True)
    bin_widths = group['bin_end_notincl'] - group['bin_start_incl']

    out = []
    for q in quantiles:
        idx_list = group.index[group['cum_val'] >= q]

        if len(idx_list) == 0:
            out.append(group.iloc[-1]['bin_end_notincl'])
            continue

        idx = idx_list[0]
        row = group.loc[idx]
        prev_cum = group.loc[idx - 1, 'cum_val'] if idx > 0 else 0.0
        bin_width = bin_widths.iloc[idx]

        if bin_width <= max_bin_width:
            if row['cum_val'] > prev_cum:
                frac = (q - prev_cum) / (row['cum_val'] - prev_cum)
                value = row['bin_start_incl'] + frac * bin_width
            else:
                value = row['bin_start_incl']
        else:
            value = _extrapolate_from_previous_bin(
                group, idx, q, bin_widths, max_bin_width
            )

        out.append(value)

    return pd.DataFrame({'quantile': quantiles, 'value': out})


def _extrapolate_from_previous_bin(group, idx, q, bin_widths, max_bin_width):
    wide_bin_start = group.loc[idx, 'bin_start_incl']
    cum_at_wide_start = group.loc[idx - 1, 'cum_val'] if idx > 0 else 0.0

    prev_idx = idx - 1
    while prev_idx >= 0 and bin_widths.iloc[prev_idx] > max_bin_width:
        prev_idx -= 1

    if prev_idx <= 0:
        return wide_bin_start

    prev_row = group.loc[prev_idx]
    cum_before_prev = group.loc[prev_idx - 1, 'cum_val'] if prev_idx > 0 else 0.0
    prev_width = prev_row['bin_end_notincl'] - prev_row['bin_start_incl']
    mass_in_prev = prev_row['cum_val'] - cum_before_prev

    if mass_in_prev <= 0 or prev_width <= 0:
        return wide_bin_start

    slope = mass_in_prev / prev_width
    value = wide_bin_start + (q - cum_at_wide_start) / slope

    return max(value, wide_bin_start)






# Ensemble

In [6]:
forecasts_all = pd.read_csv('../dat/ILIforecasts_all.csv').drop(columns='Unnamed: 0')

df = forecasts_all.copy()
df = df.sort_values(['location','forecast_date', 'horizon', 'bin_start_incl','submission_date']).copy()
df['cum_val'] = df.groupby(['location','horizon','forecast_date','submission_date'])['value'].cumsum()


result23_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date',
                      'submission_date']).apply(get_23_quantiles_interp).reset_index()).drop(columns='level_6')

result7_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date',
                      'submission_date']).apply(get_7_quantiles_interp).reset_index()).drop(columns='level_6')


result23_interp['Model'] = '23 quantiles Ensemble with Interpolation'
result7_interp['Model'] = '7 quantiles Ensemble with Interpolation'


In [ ]:
quantall_interp = pd.concat([result23_interp, result7_interp, ])

quantall_interp.to_parquet('ILI_quantile_forecasts_interpolation.parquet')

In [15]:
#bin forecasts

df2016 = forecasts_all[(forecasts_all.season!='2015-2016') ]

dfbins = pd.DataFrame()
i=0
for binstart in forecasts_all[forecasts_all.season=='2015-2016'].bin_start_incl.unique():
    binend = list(forecasts_all[forecasts_all.season=='2015-2016'].bin_end_notincl.unique())[i]
    
    df = df2016[(df2016.bin_start_incl>=binstart) & (df2016.bin_end_notincl<=binend)]
    
    dfg = df.groupby(['location','target','season','forecast_date', 'submission_date','target_end_date',
                      'horizon'])['value'].sum().reset_index()
    dfg['bin_start_incl'] = binstart
    dfg['bin_end_notincl'] = binend
    
    dfbins = pd.concat([dfbins,dfg])

    i+=1
    
bigbins = dfbins.copy()
bigbins = pd.concat([bigbins, forecasts_all[forecasts_all.season=='2015-2016']]).drop(columns=['type', 'unit'])


In [17]:
smallbins = forecasts_all[(forecasts_all.season!='2015-2016') ].drop(columns=['type', 'unit'])

In [ ]:
bigbins['Model'] = 'Bigger Bin Size Ensemble'
smallbins['Model'] = 'Smaller Bin Size Ensemble'

binall = pd.concat([bigbins, smallbins])

binall.to_parquet('ILI_binned_forecasts.parquet')

# Hindcast

In [543]:
# generate experimental hindcast

hindcasts_all = pd.read_parquet('../benchmark forecasts/hindcasts/flusight-hindcast-Bin.parquet')

df = hindcasts_all.copy()
df = df.sort_values(['location','target_end_date',  'bin_start_incl']).copy()
df['cum_val'] = df.groupby(['location','target_end_date',])['value'].cumsum()



result23_hind_interp = (df.groupby(["location", 'season', 'target_end_date']).apply(get_23_quantiles_interp).reset_index()).drop(columns='level_3')
result7_hind_interp = (df.groupby(["location", 'season', 'target_end_date']).apply(get_7_quantiles_interp).reset_index()).drop(columns='level_3')


result23_hind_interp['Model'] = '23 quantiles Hindcast with Interpolation'
result7_hind_interp['Model'] = '7 quantiles Hindcast with Interpolation'


In [ ]:
quantall_hind_interp = pd.concat([result23_hind_interp, result7_hind_interp,])

quantall_hind_interp.to_parquet('ILI_quantile_forecasts_hindcast_interpolation.parquet')

In [547]:
#bin forecasts
hindcasts_all = pd.read_parquet('../benchmark forecasts/hindcasts/flusight-hindcast-Bin.parquet')

df2016 = hindcasts_all[(hindcasts_all.season!='2015-2016') ]

dfbins = pd.DataFrame()
i=0
for binstart in hindcasts_all[hindcasts_all.season=='2015-2016'].bin_start_incl.unique():
    binend = list(hindcasts_all[hindcasts_all.season=='2015-2016'].bin_end_notincl.unique())[i]
    
    df = df2016[(df2016.bin_start_incl>=binstart) & (df2016.bin_end_notincl<=binend)]
    
    dfg = df.groupby(['location','target','season','target_end_date'])['value'].sum().reset_index()
    dfg['bin_start_incl'] = binstart
    dfg['bin_end_notincl'] = binend
    
    dfbins = pd.concat([dfbins,dfg])

    i+=1
    
bigbins_hind = dfbins.copy()
bigbins_hind = pd.concat([bigbins_hind, hindcasts_all[hindcasts_all.season=='2015-2016']])


smallbins_hind = hindcasts_all[(hindcasts_all.season!='2015-2016') ]

In [ ]:
bigbins_hind['Model'] = 'Bigger Bin Size Hindcast'
smallbins_hind['Model'] = 'Smaller Bin Size Hindcast'

binall_hind = pd.concat([bigbins_hind, smallbins_hind])

binall_hind.to_parquet('ILI_binned_forecasts_hindcast.parquet')

# Random walk baseline

In [21]:
# generate baseline

baseline_all = pd.read_parquet('../benchmark forecasts/random walk baseline/ILI_log_baseline_h12_new.pq')

baseline_all['forecast_date'] = pd.to_datetime(baseline_all['forecast_date'])

baseline_all['season'] = baseline_all['forecast_date'].apply(lambda x: f'{str(x.year)}-{str(x.year+1)}' if \
                                                     x.week>=30 else f'{str(x.year-1)}-{str(x.year)}')



df = baseline_all.copy()
df = df.sort_values(['location','forecast_date', 'horizon','target_end_date', 'bin_start_incl']).copy()
df['cum_val'] = df.groupby(['location','forecast_date','target_end_date','horizon',])['value'].cumsum()



result23_base_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date']).apply(\
                                                    get_23_quantiles_interp).reset_index()).drop(columns='level_5')
result7_base_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date']).apply(\
                                                    get_7_quantiles_interp).reset_index()).drop(columns='level_5')


result23_base_interp['Model'] = '23 quantiles Log Baseline with Interpolation'
result7_base_interp['Model'] = '7 quantiles Log Baseline with Interpolation'


In [ ]:
quantall_base_interp = pd.concat([result23_base_interp, result7_base_interp, ])

quantall_base_interp.to_parquet('ILI_quantile_forecasts_logbaseline_interpolation.parquet')

In [25]:

df2016 = baseline_all[(baseline_all.season!='2015-2016') ]

dfbins = pd.DataFrame()
i=0
for binstart in baseline_all[baseline_all.season=='2015-2016'].bin_start_incl.unique():
    binend = list(baseline_all[baseline_all.season=='2015-2016'].bin_end_notincl.unique())[i]
    
    df = df2016[(df2016.bin_start_incl>=binstart) & (df2016.bin_end_notincl<=binend)]
    
    dfg = df.groupby(['location','target','season','forecast_date', 'target_end_date',
                      'horizon'])['value'].sum().reset_index()
    dfg['bin_start_incl'] = binstart
    dfg['bin_end_notincl'] = binend
    
    dfbins = pd.concat([dfbins,dfg])

    i+=1
    
bigbins_base = dfbins.copy()
bigbins_base = pd.concat([bigbins_base, baseline_all[baseline_all.season=='2015-2016']])


smallbins_base = baseline_all[(baseline_all.season!='2015-2016') ]

In [ ]:
bigbins_base['Model'] = 'Bigger Bin Size Log Baseline'
smallbins_base['Model'] = 'Smaller Bin Size Log Baseline'

binall_base = pd.concat([bigbins_base, smallbins_base])

binall_base.to_parquet('ILI_binned_forecasts_logbaseline.parquet')

# Naive baseline

In [29]:
# generate experimental baseline

baseline_all = pd.read_parquet('../benchmark forecasts/naive/naive_flu_ili_h12_2014-20.parquet')
baseline_all['horizon'] = baseline_all['target'].apply(lambda x: int(x[0:-9])-2)
#baseline_all = baseline_all[baseline_all.horizon<=3]

baseline_all['forecast_date'] = pd.to_datetime(baseline_all['forecast_date'])

baseline_all['season'] = baseline_all['forecast_date'].apply(lambda x: f'{str(x.year)}-{str(x.year+1)}' if \
                                                     x.week>=30 else f'{str(x.year-1)}-{str(x.year)}')



df = baseline_all.copy()
df = df.sort_values(['location','forecast_date', 'horizon','target_end_date', 'bin_start_incl']).copy()
df['cum_val'] = df.groupby(['location','horizon','forecast_date','target_end_date'])['value'].cumsum()



result23_naive_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date']).apply(\
                                                    get_23_quantiles_interp).reset_index()).drop(columns='level_5')
result7_naive_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date']).apply(\
                                                   get_7_quantiles_interp).reset_index()).drop(columns='level_5')


result23_naive_interp['Model'] = '23 quantiles Naive with Interpolation'
result7_naive_interp['Model'] = '7 quantiles Naive with Interpolation'


In [ ]:
quantall_naive_interp = pd.concat([result23_naive_interp, result7_naive_interp, ])

quantall_naive_interp.to_parquet('ILI_quantile_forecasts_naive_interpolation.parquet')

In [33]:
df2016 = baseline_all[(baseline_all.season!='2015-2016') ]

dfbins = pd.DataFrame()
i=0
for binstart in baseline_all[baseline_all.season=='2015-2016'].bin_start_incl.unique():
    binend = list(baseline_all[baseline_all.season=='2015-2016'].bin_end_notincl.unique())[i]
    
    df = df2016[(df2016.bin_start_incl>=binstart) & (df2016.bin_end_notincl<=binend)]
    
    dfg = df.groupby(['location','target','season','forecast_date', 'target_end_date',
                      'horizon'])['value'].sum().reset_index()
    dfg['bin_start_incl'] = binstart
    dfg['bin_end_notincl'] = binend
    
    dfbins = pd.concat([dfbins,dfg])

    i+=1
    
bigbins_naive = dfbins.copy()
bigbins_naive = pd.concat([bigbins_naive, baseline_all[baseline_all.season=='2015-2016']])


smallbins_naive = baseline_all[(baseline_all.season!='2015-2016') ]

In [ ]:
bigbins_naive['Model'] = 'Bigger Bin Size Naive'
smallbins_naive['Model'] = 'Smaller Bin Size Naive'

binall_naive = pd.concat([bigbins_naive, smallbins_naive])

binall_naive.to_parquet('ILI_binned_forecasts_naive.parquet')

# Hist-Avg

In [38]:
# generate experimental baseline

baseline_all = pd.read_csv('../dat/ILIforecasts_Hist-Avg.csv')

baseline_all['forecast_date'] = pd.to_datetime(baseline_all['forecast_date'])


df = baseline_all.copy()
df = df.sort_values(['location','forecast_date', 'horizon', 'bin_start_incl','submission_date']).copy()
df['cum_val'] = df.groupby(['location','horizon','forecast_date','submission_date'])['value'].cumsum()


result23_hist_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date', 'submission_date']).apply(\
                                                    get_23_quantiles_interp).reset_index()).drop(columns='level_6')
result7_hist_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date', 'submission_date']).apply(\
                                                    get_7_quantiles_interp).reset_index()).drop(columns='level_6')


result23_hist_interp['Model'] = '23 quantiles Hist-Avg with Interpolation'
result7_hist_interp['Model'] = '7 quantiles Hist-Avg with Interpolation'


In [ ]:
quantall_hist_interp = pd.concat([result23_hist_interp, result7_hist_interp, ])

quantall_hist_interp.to_parquet('ILI_quantile_forecasts_Hist-Avg_interpolation.parquet')

In [41]:
df2016 = baseline_all[(baseline_all.season!='2015-2016') ]

dfbins = pd.DataFrame()
i=0
for binstart in baseline_all[baseline_all.season=='2015-2016'].bin_start_incl.unique():
    binend = list(baseline_all[baseline_all.season=='2015-2016'].bin_end_notincl.unique())[i]
    
    df = df2016[(df2016.bin_start_incl>=binstart) & (df2016.bin_end_notincl<=binend)]
    
    dfg = df.groupby(['location','target','season','forecast_date', 'target_end_date',
                      'horizon'])['value'].sum().reset_index()
    dfg['bin_start_incl'] = binstart
    dfg['bin_end_notincl'] = binend
    
    dfbins = pd.concat([dfbins,dfg])

    i+=1
    
bigbins_hist = dfbins.copy()
bigbins_hist = pd.concat([bigbins_hist, baseline_all[baseline_all.season=='2015-2016']])


smallbins_hist = baseline_all[(baseline_all.season!='2015-2016') ]

In [ ]:
bigbins_hist['Model'] = 'Bigger Bin Size Hist-Avg'
smallbins_hist['Model'] = 'Smaller Bin Size Hist-Avg'

binall_hist = pd.concat([bigbins_hist, smallbins_hist])

binall_hist.to_parquet('ILI_binned_forecasts_Hist-Avg.parquet')

# Trend baseline

In [46]:
# generate experimental baseline

#baseline_all = pd.read_parquet('./log baseline forecasts/ILI_log_baseline_h12.pq')
baseline_all = pd.read_parquet('../benchmark forecasts/trend baseline/ets_additive_damped_ili_logit_2003on_bins_2014-20.parquet')
baseline_all['horizon'] = baseline_all['target'].apply(lambda x: int(x[0:-9])-2)
#baseline_all = baseline_all[baseline_all.horizon<=3]

baseline_all['forecast_date'] = pd.to_datetime(baseline_all['forecast_date'])

baseline_all['season'] = baseline_all['forecast_date'].apply(lambda x: f'{str(x.year)}-{str(x.year+1)}' if \
                                                     x.week>=30 else f'{str(x.year-1)}-{str(x.year)}')



df = baseline_all.copy()
df = df.sort_values(['location','forecast_date', 'horizon','target_end_date', 'bin_start_incl']).copy()
df['cum_val'] = df.groupby(['location','horizon','forecast_date','target_end_date'])['value'].cumsum()



result23_ETS_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date']).apply(\
                                                    get_23_quantiles_interp).reset_index()).drop(columns='level_5')
result7_ETS_interp = (df.groupby(["location", "horizon", "forecast_date", 'season', 'target_end_date']).apply(\
                                                    get_7_quantiles_interp).reset_index()).drop(columns='level_5')

result23_ETS_interp['Model'] = '23 quantiles ETS with Interpolation'
result7_ETS_interp['Model'] = '7 quantiles ETS with Interpolation'


In [ ]:
quantall_ETS_interp = pd.concat([result23_ETS_interp, result7_ETS_interp, ])

quantall_ETS_interp.to_parquet('ILI_quantile_forecasts_ETS_interpolation.parquet')

In [49]:
df2016 = baseline_all[(baseline_all.season!='2015-2016') ]

dfbins = pd.DataFrame()
i=0
for binstart in baseline_all[baseline_all.season=='2015-2016'].bin_start_incl.unique():
    binend = list(baseline_all[baseline_all.season=='2015-2016'].bin_end_notincl.unique())[i]
    
    df = df2016[(df2016.bin_start_incl>=binstart) & (df2016.bin_end_notincl<=binend)]
    
    dfg = df.groupby(['location','target','season','forecast_date', 'target_end_date',
                      'horizon'])['value'].sum().reset_index()
    dfg['bin_start_incl'] = binstart
    dfg['bin_end_notincl'] = binend
    
    dfbins = pd.concat([dfbins,dfg])

    i+=1
    
bigbins_ets = dfbins.copy()
bigbins_ets = pd.concat([bigbins_ets, baseline_all[baseline_all.season=='2015-2016']])


smallbins_ets = baseline_all[(baseline_all.season!='2015-2016') ]

In [ ]:
bigbins_ets['Model'] = 'Bigger Bin Size ETS'
smallbins_ets['Model'] = 'Smaller Bin Size ETS'

binall_ets = pd.concat([bigbins_ets, smallbins_ets])

binall_ets.to_parquet('ILI_binned_forecasts_ETS.parquet')

# Score quantiles (WIS)

In [8]:

_IDX = ['Model', 'location', 'horizon', 'reference_date', 'target_end_date']


class scoring_functions:

    def interval_score(self, observation, lower, upper, interval_range):
        if len(lower) != len(upper) or len(lower) != len(observation):
            raise ValueError("vector shape mismatch")
        if interval_range > 100 or interval_range < 0:
            raise ValueError("interval range should be between 0 and 100")
        obs, l, u = np.array(observation), np.array(lower), np.array(upper)
        alpha = 1 - interval_range / 100
        dispersion = u - l
        overprediction = (2 / alpha) * (l - obs) * (obs < l)
        underprediction = (2 / alpha) * (obs - u) * (obs > u)
        score = dispersion + underprediction + overprediction
        return {'interval_score': score, 'dispersion': dispersion,
                'underprediction': underprediction, 'overprediction': overprediction}

    def timestamp_wis(self, observations, predsfilt,
                      interval_ranges=[10,20,30,40,50,60,70,80,90,95,98]):
        quantiles = np.array(predsfilt.sort_values(by='output_type_id').output_type_id, dtype=float)
        qs = []
        for q in quantiles:
            df = predsfilt[predsfilt.output_type_id == q].sort_values(by='target_end_date')
            qs.append(np.array(df.value))
        Q = np.array(qs)
        y = np.array(observations.value)

        # find median by value, not by position
        med_idx = np.argmin(np.abs(quantiles - 0.5))
        if not np.isclose(quantiles[med_idx], 0.5):
            raise ValueError("median (0.5) not present")

        WIS = np.zeros(len(y))
        n_pairs = 0
        for i in range(len(quantiles) // 2):
            if np.isclose(quantiles[i] + quantiles[-i-1], 1.0):   # symmetric pair
                interval_range = 100 * (quantiles[-i-1] - quantiles[i])
                alpha = 1 - (quantiles[-i-1] - quantiles[i])
                IS = self.interval_score(y, Q[i], Q[-i-1], interval_range)
                WIS += IS['interval_score'] * alpha / 2
                n_pairs += 1
        WIS += 0.5 * np.abs(Q[med_idx] - y)
        WIS = WIS / (n_pairs + 0.5)          # <-- fixed normalisation

        return pd.DataFrame({
            'Model': predsfilt.Model.unique(), 'location': predsfilt.location.unique(),
            'horizon': predsfilt.horizon.unique(), 'reference_date': predsfilt.reference_date.unique(),
            'target_end_date': predsfilt.target_end_date.unique(), 'wis': WIS[0]}, index=[0])

    def get_wis_scores_vectorized(self, predsall, surv, horizons=[0, 1, 2, 3]):
        obs = (surv[['location', 'date', 'value']]
               .rename(columns={'date': 'target_end_date', 'value': 'y'})).copy()
        obs['target_end_date'] = pd.to_datetime(obs['target_end_date'])

        preds = predsall[(predsall.target_end_date <= surv.date.max())
                         & (predsall.horizon.isin(horizons))].copy()
        preds['output_type_id'] = preds['output_type_id'].astype(float)
        preds['target_end_date'] = pd.to_datetime(preds['target_end_date'])

        df = preds.merge(obs, on=['location', 'target_end_date'], how='inner')

        wide = df.pivot_table(index=_IDX, columns='output_type_id', values='value', aggfunc='first')
        y = df.groupby(_IDX)['y'].first().reindex(wide.index)

        # drop tasks missing any submitted quantile level
        n_before = len(wide)
        wide = wide.dropna(axis=0, how='any')
        if n_before - len(wide):
            print(f"get_wis_scores_vectorized: dropped {n_before - len(wide)} task(s) with missing quantiles")
        y = y.reindex(wide.index).to_numpy(dtype=float)

        levels = {round(float(c), 5): c for c in wide.columns}
        if 0.5 not in levels:
            raise ValueError("median (0.5) must be present")
        m = wide[levels[0.5]].to_numpy(dtype=float)

        disp = np.zeros(len(wide)); under = np.zeros(len(wide)); over = np.zeros(len(wide))
        K = 0
        for q_lo in sorted(l for l in levels if l < 0.5):
            q_hi = round(1 - q_lo, 5)
            if q_hi not in levels:
                continue
            alpha = 1 - (q_hi - q_lo)
            l = wide[levels[q_lo]].to_numpy(dtype=float)
            u = wide[levels[q_hi]].to_numpy(dtype=float)
            disp += 0.5 * alpha * (u - l)
            over += np.maximum(l - y, 0.0)      
            under += np.maximum(y - u, 0.0)       
            K += 1
        over += 0.5 * np.maximum(m - y, 0.0)    # median term
        under += 0.5 * np.maximum(y - m, 0.0)
        denom = K + 0.5

        dfwis = wide.index.to_frame(index=False)
        dfwis['wis'] = (disp + under + over) / denom
        
        return dfwis

    # ---- old entry point, same signature, now vectorized ----------------
    def get_wis_scores(self, predsall, surv, models=False, dates=False,
                       save_location=False, horizons=[0, 1, 2, 3], components=False):
        preds = predsall
        if models is not False and models is not None:
            preds = preds[preds.Model.isin(models)]
        if dates is not False and dates is not None:
            preds = preds[preds.reference_date.isin(dates)]
        return self.get_wis_scores_vectorized(preds, surv, save_location=save_location,
                                              horizons=horizons, components=components)
    

In [ ]:
## score ensemble

In [9]:
sf = scoring_functions()

dfwisall = pd.DataFrame()
for dfq in [result23_interp, result7_interp]:
    print(dfq.Model.unique()[0])

    observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[
        ['region','year','week','weighted_ili','week_start','location']]
    observations['week_start'] = pd.to_datetime(observations['week_start'])
    observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
    observations['obs_value'] = observations['weighted_ili'] / 100
    observations = observations.drop(columns=['region','weighted_ili'])
    observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])

    surv = observations.copy()
    surv['date']  = pd.to_datetime(surv['target_end_date'])   # scorer needs `date`
    surv = surv.rename(columns={'obs_value':'value'})          # scorer needs `value`

    predsall = dfq.copy()
    predsall['target_end_date'] = pd.to_datetime(predsall['target_end_date'])
    if 'output_type' in predsall.columns:                      # drop point rows
        predsall = predsall[predsall['output_type'] == 'quantile']
    predsall = predsall.rename(columns={'quantile':'output_type_id'})   # quantile-level column
    predsall['output_type_id'] = predsall['output_type_id'].astype(float)
    predsall['reference_date'] = predsall['submission_date']   # task key = the submission

    predsall['value'] = predsall['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))
    surv['value']     = surv['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))

    dfwis = sf.get_wis_scores_vectorized(predsall, surv, horizons=[-1, 0, 1, 2])

    fmap = predsall.drop_duplicates('submission_date').set_index('submission_date')['forecast_date']
    dfwis = dfwis.rename(columns={'reference_date':'submission_date'})
    dfwis['forecast_date'] = dfwis['submission_date'].map(fmap)

    dfwisall = pd.concat([dfwisall, dfwis])

dfwisall = dfwisall.rename(columns={'wis':'log_wis'})

23 quantiles Ensemble with Interpolation
7 quantiles Ensemble with Interpolation


In [19]:
dfwisall = dfwisall.rename(columns={'log_wis':'logit_wis'})

In [ ]:
dfwisall.to_parquet('scores/ensemble_logit_wis.parquet')

In [388]:
### score hindcast quantiles


In [ ]:
sf = scoring_functions()

dfwis_hind= pd.DataFrame()
for dfq in [result23_hind_interp, result7_hind_interp]:
    print(dfq.Model.unique()[0])

    observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[
        ['region','year','week','weighted_ili','week_start','location']]
    observations['week_start'] = pd.to_datetime(observations['week_start'])
    observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
    observations['obs_value'] = observations['weighted_ili'] / 100
    observations = observations.drop(columns=['region','weighted_ili'])
    observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])

    surv = observations.copy()
    surv['date']  = pd.to_datetime(surv['target_end_date'])   # scorer needs `date`
    surv = surv.rename(columns={'obs_value':'value'})          # scorer needs `value`

    predsall = dfq.copy()
    predsall['target_end_date'] = pd.to_datetime(predsall['target_end_date'])
    if 'output_type' in predsall.columns:                      # drop point rows
        predsall = predsall[predsall['output_type'] == 'quantile']
    predsall = predsall.rename(columns={'quantile':'output_type_id'})   # quantile-level column
    predsall['output_type_id'] = predsall['output_type_id'].astype(float)
    predsall['reference_date'] = predsall['submission_date']   # task key = the submission

    predsall['value'] = predsall['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))
    surv['value']     = surv['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))

    dfwis = sf.get_wis_scores_vectorized(predsall, surv, horizons=[-1, 0, 1, 2])

    fmap = predsall.drop_duplicates('submission_date').set_index('submission_date')['forecast_date']
    dfwis = dfwis.rename(columns={'reference_date':'submission_date'})
    dfwis['forecast_date'] = dfwis['submission_date'].map(fmap)

    dfwis_hind = pd.concat([dfwis_hind, dfwis])

dfwis_hind = dfwis_hind.rename(columns={'wis':'logit_wis'})

In [ ]:
dfwis_hind.to_parquet('scores/hindcast_logit_wis.parquet')

In [389]:
### score RW baseline quantiles

In [ ]:
sf = scoring_functions()

dfwis_base= pd.DataFrame()
for dfq in [result23_base_interp, result7_base_interp]:
    print(dfq.Model.unique()[0])

    observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[
        ['region','year','week','weighted_ili','week_start','location']]
    observations['week_start'] = pd.to_datetime(observations['week_start'])
    observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
    observations['obs_value'] = observations['weighted_ili'] / 100
    observations = observations.drop(columns=['region','weighted_ili'])
    observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])

    surv = observations.copy()
    surv['date']  = pd.to_datetime(surv['target_end_date'])   # scorer needs `date`
    surv = surv.rename(columns={'obs_value':'value'})          # scorer needs `value`

    predsall = dfq.copy()
    predsall['target_end_date'] = pd.to_datetime(predsall['target_end_date'])
    if 'output_type' in predsall.columns:                      # drop point rows
        predsall = predsall[predsall['output_type'] == 'quantile']
    predsall = predsall.rename(columns={'quantile':'output_type_id'})   # quantile-level column
    predsall['output_type_id'] = predsall['output_type_id'].astype(float)
    predsall['reference_date'] = predsall['submission_date']   # task key = the submission

    predsall['value'] = predsall['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))
    surv['value']     = surv['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))

    dfwis = sf.get_wis_scores_vectorized(predsall, surv, horizons=[-1, 0, 1, 2])

    fmap = predsall.drop_duplicates('submission_date').set_index('submission_date')['forecast_date']
    dfwis = dfwis.rename(columns={'reference_date':'submission_date'})
    dfwis['forecast_date'] = dfwis['submission_date'].map(fmap)

    dfwis_base = pd.concat([dfwis_base, dfwis])

dfwis_base = dfwis_base.rename(columns={'wis':'logit_wis'})

In [ ]:
dfwis_base.to_parquet('scores/logbaseline_logit_wis.parquet')

In [ ]:
## score naive baseline 

In [ ]:
sf = scoring_functions()

dfwis_naive= pd.DataFrame()
for dfq in [result23_naive_interp, result7_naive_interp]:
    print(dfq.Model.unique()[0])

    observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[
        ['region','year','week','weighted_ili','week_start','location']]
    observations['week_start'] = pd.to_datetime(observations['week_start'])
    observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
    observations['obs_value'] = observations['weighted_ili'] / 100
    observations = observations.drop(columns=['region','weighted_ili'])
    observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])

    surv = observations.copy()
    surv['date']  = pd.to_datetime(surv['target_end_date'])   # scorer needs `date`
    surv = surv.rename(columns={'obs_value':'value'})          # scorer needs `value`

    predsall = dfq.copy()
    predsall['target_end_date'] = pd.to_datetime(predsall['target_end_date'])
    if 'output_type' in predsall.columns:                      # drop point rows
        predsall = predsall[predsall['output_type'] == 'quantile']
    predsall = predsall.rename(columns={'quantile':'output_type_id'})   # quantile-level column
    predsall['output_type_id'] = predsall['output_type_id'].astype(float)
    predsall['reference_date'] = predsall['submission_date']   # task key = the submission

    predsall['value'] = predsall['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))
    surv['value']     = surv['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))

    dfwis = sf.get_wis_scores_vectorized(predsall, surv, horizons=[-1, 0, 1, 2])

    fmap = predsall.drop_duplicates('submission_date').set_index('submission_date')['forecast_date']
    dfwis = dfwis.rename(columns={'reference_date':'submission_date'})
    dfwis['forecast_date'] = dfwis['submission_date'].map(fmap)

    dfwis_naive = pd.concat([dfwis_naive, dfwis])

dfwis_naive = dfwis_naive.rename(columns={'wis':'logit_wis'})

In [ ]:
dfwis_naive.to_parquet('scores/naive_logit_wis.parquet')

In [ ]:
## score hub baseline (Hist-Avg) quantiles

In [ ]:
sf = scoring_functions()

dfwis_hist= pd.DataFrame()
for dfq in [result23_hist_interp, result7_hist_interp]:
    print(dfq.Model.unique()[0])

    observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[
        ['region','year','week','weighted_ili','week_start','location']]
    observations['week_start'] = pd.to_datetime(observations['week_start'])
    observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
    observations['obs_value'] = observations['weighted_ili'] / 100
    observations = observations.drop(columns=['region','weighted_ili'])
    observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])

    surv = observations.copy()
    surv['date']  = pd.to_datetime(surv['target_end_date'])   # scorer needs `date`
    surv = surv.rename(columns={'obs_value':'value'})          # scorer needs `value`

    predsall = dfq.copy()
    predsall['target_end_date'] = pd.to_datetime(predsall['target_end_date'])
    if 'output_type' in predsall.columns:                      # drop point rows
        predsall = predsall[predsall['output_type'] == 'quantile']
    predsall = predsall.rename(columns={'quantile':'output_type_id'})   # quantile-level column
    predsall['output_type_id'] = predsall['output_type_id'].astype(float)
    predsall['reference_date'] = predsall['submission_date']   # task key = the submission

    predsall['value'] = predsall['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))
    surv['value']     = surv['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))

    dfwis = sf.get_wis_scores_vectorized(predsall, surv, horizons=[-1, 0, 1, 2])

    fmap = predsall.drop_duplicates('submission_date').set_index('submission_date')['forecast_date']
    dfwis = dfwis.rename(columns={'reference_date':'submission_date'})
    dfwis['forecast_date'] = dfwis['submission_date'].map(fmap)

    dfwis_hist = pd.concat([dfwis_hist, dfwis])

dfwis_hist = dfwis_hist.rename(columns={'wis':'logit_wis'})

In [ ]:
dfwis_hist.to_parquet('scores/Hist-Avg_logit_wis.parquet')

In [ ]:
## score trend baseline quantiles

In [ ]:
sf = scoring_functions()

dfwis_ets= pd.DataFrame()
for dfq in [result23_ets_interp, result7_ets_interp]:
    print(dfq.Model.unique()[0])

    observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[
        ['region','year','week','weighted_ili','week_start','location']]
    observations['week_start'] = pd.to_datetime(observations['week_start'])
    observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
    observations['obs_value'] = observations['weighted_ili'] / 100
    observations = observations.drop(columns=['region','weighted_ili'])
    observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])

    surv = observations.copy()
    surv['date']  = pd.to_datetime(surv['target_end_date'])   # scorer needs `date`
    surv = surv.rename(columns={'obs_value':'value'})          # scorer needs `value`

    predsall = dfq.copy()
    predsall['target_end_date'] = pd.to_datetime(predsall['target_end_date'])
    if 'output_type' in predsall.columns:                      # drop point rows
        predsall = predsall[predsall['output_type'] == 'quantile']
    predsall = predsall.rename(columns={'quantile':'output_type_id'})   # quantile-level column
    predsall['output_type_id'] = predsall['output_type_id'].astype(float)
    predsall['reference_date'] = predsall['submission_date']   # task key = the submission

    predsall['value'] = predsall['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))
    surv['value']     = surv['value'].apply(lambda x: logit(x+0.001) if x==0 else logit(x))

    dfwis = sf.get_wis_scores_vectorized(predsall, surv, horizons=[-1, 0, 1, 2])

    fmap = predsall.drop_duplicates('submission_date').set_index('submission_date')['forecast_date']
    dfwis = dfwis.rename(columns={'reference_date':'submission_date'})
    dfwis['forecast_date'] = dfwis['submission_date'].map(fmap)

    dfwis_ets = pd.concat([dfwis_ets, dfwis])

dfwis_ets = dfwis_ets.rename(columns={'wis':'logit_wis'})

In [ ]:
dfwis_ets.to_parquet('scores/ETS_logit_wis.parquet')

# Score bins (Log score)

In [81]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
bigbins['target_end_date'] = pd.to_datetime(bigbins['target_end_date'])

observations = observations[observations.target_end_date.isin(bigbins.target_end_date.unique())]

ls_baseline = bigbins.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['forecast_date', 'horizon' ,'location', 'target', 'target_end_date', 'ls', 'season']]

ls_bigbins = ls_baseline.copy()
ls_bigbins['Model'] = 'Bigger Bins'

In [82]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
smallbins['target_end_date'] = pd.to_datetime(smallbins['target_end_date'])

observations = observations[observations.target_end_date.isin(smallbins.target_end_date.unique())]

ls_baseline = smallbins.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['forecast_date', 'horizon' ,'location', 'target', 'target_end_date', 'ls', 'season']]

ls_smallbins = ls_baseline.copy()
ls_smallbins['Model'] = 'Smaller Bins'

In [83]:
ls_all = pd.concat([ls_smallbins, ls_bigbins])

In [ ]:
ls_all.to_parquet('scores/ensemble_binned_LS.parquet')

In [948]:
# score hindcast

In [485]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
bigbins_hind['target_end_date'] = pd.to_datetime(bigbins_hind['target_end_date'])

observations = observations[observations.target_end_date.isin(bigbins.target_end_date.unique())]

ls_baseline = bigbins_hind.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season']]

ls_bigbins_hind = ls_baseline.copy()
ls_bigbins_hind['Model'] = 'Bigger Bins Hindcast'

In [486]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
smallbins_hind['target_end_date'] = pd.to_datetime(smallbins_hind['target_end_date'])

observations = observations[observations.target_end_date.isin(bigbins.target_end_date.unique())]

ls_baseline = smallbins_hind.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season']]

ls_smallbins_hind = ls_baseline.copy()
ls_smallbins_hind['Model'] = 'Smaller Bins Hindcast'

In [ ]:
ls_hind = pd.concat([ls_bigbins_hind, ls_smallbins_hind])

ls_hind.to_parquet('scores/hindcast_binned_LS.parquet')

In [951]:
# score RW baseline model

In [87]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
bigbins_base['target_end_date'] = pd.to_datetime(bigbins_base['target_end_date'])

observations = observations[observations.target_end_date.isin(bigbins_base.target_end_date.unique())]

ls_baseline = bigbins_base.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season',  'horizon', 'forecast_date']]

ls_bigbins_base = ls_baseline.copy()
ls_bigbins_base['Model'] = 'Bigger Bins Baseline'

In [88]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
smallbins_base['target_end_date'] = pd.to_datetime(smallbins_base['target_end_date'])

observations = observations[observations.target_end_date.isin(smallbins_base.target_end_date.unique())]

ls_baseline = smallbins_base.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season', 'horizon', 'forecast_date']]

ls_smallbins_base = ls_baseline.copy()
ls_smallbins_base['Model'] = 'Smaller Bins Baseline'

In [ ]:
ls_base = pd.concat([ls_bigbins_base, ls_smallbins_base])

ls_base.to_parquet('scores/logbaseline_binned_LS.parquet')

In [ ]:
## Naive

In [93]:
bigbins_naive['target_end_date'] = pd.to_datetime(bigbins_naive['target_end_date'])

observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
bigbins_naive['target_end_date'] = pd.to_datetime(bigbins_naive['target_end_date'])

observations = observations[observations.target_end_date.isin(bigbins_naive.target_end_date.unique())]

ls_baseline = bigbins_naive.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season',  'horizon', 'forecast_date']]

ls_bigbins_naive = ls_baseline.copy()
ls_bigbins_naive['Model'] = 'Bigger Bins Naive'

In [94]:

observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
smallbins_naive['target_end_date'] = pd.to_datetime(smallbins_naive['target_end_date'])

observations = observations[observations.target_end_date.isin(smallbins_naive.target_end_date.unique())]

ls_baseline = smallbins_naive.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season', 'horizon', 'forecast_date']]

ls_smallbins_naive = ls_baseline.copy()
ls_smallbins_naive['Model'] = 'Smaller Bins Naive'

In [ ]:
ls_naive = pd.concat([ls_bigbins_naive, ls_smallbins_naive])

ls_naive.to_parquet('scores/naive_binned_LS.parquet')

In [1]:
## Hist-Avg

In [97]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
bigbins_hist['target_end_date'] = pd.to_datetime(bigbins_hist['target_end_date'])

observations = observations[observations.target_end_date.isin(bigbins_hist.target_end_date.unique())]

ls_baseline = bigbins_hist.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season',  'horizon', 'forecast_date']]

ls_bigbins_hist = ls_baseline.copy()
ls_bigbins_hist['Model'] = 'Bigger Bins Hist-Avg'

In [98]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
smallbins_hist['target_end_date'] = pd.to_datetime(smallbins_hist['target_end_date'])

observations = observations[observations.target_end_date.isin(smallbins_hist.target_end_date.unique())]

ls_baseline = smallbins_hist.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season', 'horizon', 'forecast_date']]

ls_smallbins_hist = ls_baseline.copy()
ls_smallbins_hist['Model'] = 'Smaller Bins Hist-Avg'

In [ ]:
ls_hist = pd.concat([ls_bigbins_hist, ls_smallbins_hist])

ls_hist.to_parquet('scores/Hist-Avg_binned_LS.parquet')

In [2]:
# trend baseline

In [101]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
bigbins_ets['target_end_date'] = pd.to_datetime(bigbins_ets['target_end_date'])

observations = observations[observations.target_end_date.isin(bigbins_ets.target_end_date.unique())]

ls_baseline = bigbins_ets.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season',  'horizon', 'forecast_date']]

ls_bigbins_ets = ls_baseline.copy()
ls_bigbins_ets['Model'] = 'Bigger Bins ETS'

In [102]:
observations = pd.read_parquet('../surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]
observations['week_start'] = pd.to_datetime(observations['week_start'])
observations['target_end_date'] = observations['week_start'].apply(lambda x: Week.fromdate(x).enddate())
observations['obs_value'] = observations['weighted_ili'] / 100

observations = observations.drop(columns=['region', 'weighted_ili'])

observations['target_end_date'] = pd.to_datetime(observations['target_end_date'])
smallbins_ets['target_end_date'] = pd.to_datetime(smallbins_ets['target_end_date'])

observations = observations[observations.target_end_date.isin(smallbins_ets.target_end_date.unique())]

ls_baseline = smallbins_ets.merge(observations, how='left', on=['location', 'target_end_date'])

ls_baseline = ls_baseline[(ls_baseline['bin_start_incl'] < (ls_baseline['obs_value'])) & 
    ((ls_baseline['obs_value']) <= ls_baseline['bin_end_notincl'])]

# Create ls column (log of value, with minimum of -10)
ls_baseline['ls'] = ls_baseline['value'].apply(lambda x: max(np.log(x), -10))

# Select specific columns
ls_baseline = ls_baseline[['location', 'target', 'target_end_date', 'ls', 'season', 'horizon', 'forecast_date']]

ls_smallbins_ets = ls_baseline.copy()
ls_smallbins_ets['Model'] = 'Smaller Bins ETS'

In [ ]:
ls_ets = pd.concat([ls_bigbins_ets, ls_smallbins_ets])

ls_ets.to_parquet('scores/ETS_binned_LS.parquet')

# Combined score files

In [ ]:
dfwis_base = pd.read_parquet('scores/logbaseline_logit_wis.parquet')
dfwis_naive = pd.read_parquet('scores/naive_logit_wis.parquet')
dfwis_hist = pd.read_parquet('scores/Hist-Avg_logit_wis.parquet')

dfwis_hind = pd.read_parquet('scores/hindcast_logit_wis.parquet')
dfwis_ets = pd.read_parquet('scores/ETS_logit_wis.parquet')

dfwis = pd.read_parquet('scores/ensemble_logit_wis.parquet')


#dfwis_hind['exp_type'] = dfwis_hind['Model'].apply(lambda x: x[0:-9])
dfwis_hind['Model'] = dfwis_hind['Model'].str.replace('Hindcast', '')
dfwis_hind = dfwis_hind[['location', 'target_end_date', 'logit_wis', 'Model']].rename(columns=\
                                       {'logit_wis':'logit_wis_hindcast'})

dfwis_base['Model'] = dfwis_base['Model'].str.replace('Log Baseline', '')
dfwis_base = dfwis_base[['location', 'target_end_date', 'logit_wis', 'Model','horizon', 'forecast_date']].rename(columns=\
                                        {'logit_wis':'logit_wis_lbaseline'})

dfwis_naive['Model'] = dfwis_naive['Model'].str.replace('Naive', '')
dfwis_naive = dfwis_naive[['location', 'target_end_date', 'logit_wis', 'Model','horizon', 'forecast_date']].rename(columns=\
                                        {'logit_wis':'logit_wis_naive'})
dfwis_hist['Model'] = dfwis_hist['Model'].str.replace('Hist-Avg', '')
dfwis_hist = dfwis_hist[['location', 'target_end_date', 'logit_wis', 'Model','horizon', 'forecast_date']].rename(columns=\
                                        {'logit_wis':'logit_wis_Hist-Avg'})


dfwis_ets['Model'] = dfwis_ets['Model'].str.replace('ETS', '')
dfwis_ets = dfwis_ets[['location', 'target_end_date', 'logit_wis', 'Model','horizon', 'forecast_date']].rename(columns=\
                                        {'logit_wis':'logit_wis_ets_baseline'})



dfwis['Model'] = dfwis['Model'].str.replace('Ensemble', '')

dfwis['forecast_date'] = pd.to_datetime(dfwis['forecast_date'])
dfwis_base['forecast_date'] = pd.to_datetime(dfwis_base['forecast_date'])

dfwis['target_end_date'] = pd.to_datetime(dfwis['target_end_date'])
dfwis_base['target_end_date'] = pd.to_datetime(dfwis_base['target_end_date'])
dfwis_naive['target_end_date'] = pd.to_datetime(dfwis_naive['target_end_date'])
dfwis_hist['target_end_date'] = pd.to_datetime(dfwis_hist['target_end_date'])
dfwis_ets['target_end_date'] = pd.to_datetime(dfwis_ets['target_end_date'])



dfwisall = dfwis.merge(dfwis_hind, on=['location', 'target_end_date', 'Model'])
dfwisall = dfwisall.merge(dfwis_base, on=['location', 'target_end_date', 'Model', 'horizon', 'forecast_date'])
dfwisall = dfwisall.merge(dfwis_naive, on=['location', 'target_end_date', 'Model', 'horizon', 'forecast_date'])
dfwisall = dfwisall.merge(dfwis_hist, on=['location', 'target_end_date', 'Model', 'horizon', 'forecast_date'])
dfwisall = dfwisall.merge(dfwis_ets, on=['location', 'target_end_date', 'Model', 'horizon', 'forecast_date'])




dfwisall['season'] = dfwisall['forecast_date'].apply(lambda x: f'{str(x.year)}-{str(x.year+1)}' if \
                                                     x.week>=30 else f'{str(x.year-1)}-{str(x.year)}')

dfwisall.to_parquet('scores/logit_wis_combined.parquet')

In [ ]:
ls_base = pd.read_parquet('scores/logbaseline_binned_LS.parquet')
ls_naive = pd.read_parquet('scores/naive_binned_LS.parquet')
ls_hist = pd.read_parquet('scores/Hist-Avg_binned_LS.parquet')

ls_hind = pd.read_parquet('scores/hindcast_binned_LS.parquet')
ls_ets = pd.read_parquet('scores/ETS_binned_LS.parquet')

ls = pd.read_parquet('scores/ensemble_binned_LS.parquet')


ls_hind['exp_type'] = ls_hind['Model'].apply(lambda x: x[0:-9])
ls_hind = ls_hind[['location', 'target_end_date', 'ls', 'exp_type']].rename(columns=\
                                       {'exp_type':'Model','ls':'ls_hindcast'})

ls_base['exp_type'] = ls_base['Model'].apply(lambda x: x[0:-9])
ls_base = ls_base[['location', 'target_end_date', 'ls', 'exp_type','horizon', 'forecast_date']].rename(columns=\
                                        {'exp_type':'Model','ls':'ls_lbaseline'})

ls_naive['exp_type'] = ls_naive['Model'].apply(lambda x: x[0:-6])
ls_naive = ls_naive[['location', 'target_end_date', 'ls', 'exp_type','horizon', 'forecast_date']].rename(columns=\
                                       {'exp_type':'Model','ls':'ls_naive'})

ls_hist['exp_type'] = ls_hist['Model'].apply(lambda x: x[0:-9])
ls_hist = ls_hist[['location', 'target_end_date', 'ls', 'exp_type','horizon', 'forecast_date']].rename(columns=\
                                       {'exp_type':'Model','ls':'ls_Hist-Avg'})

ls_ets['exp_type'] = ls_ets['Model'].apply(lambda x: x[0:-4])
ls_ets = ls_ets[['location', 'target_end_date', 'ls', 'exp_type','horizon', 'forecast_date']].rename(columns=\
                                       {'exp_type':'Model','ls':'ls_ETS'})


ls['forecast_date'] = pd.to_datetime(ls['forecast_date'])
ls_base['forecast_date'] = pd.to_datetime(ls_base['forecast_date'])
ls_naive['forecast_date'] = pd.to_datetime(ls_naive['forecast_date'])
ls_hist['forecast_date'] = pd.to_datetime(ls_hist['forecast_date'])
ls_ets['forecast_date'] = pd.to_datetime(ls_ets['forecast_date'])

ls['target_end_date'] = pd.to_datetime(ls['target_end_date'])
ls_base['target_end_date'] = pd.to_datetime(ls_base['target_end_date'])
ls_naive['target_end_date'] = pd.to_datetime(ls_naive['target_end_date'])
ls_hist['target_end_date'] = pd.to_datetime(ls_hist['target_end_date'])
ls_ets['target_end_date'] = pd.to_datetime(ls_ets['target_end_date'])



ls_all = ls.merge(ls_hind, on=['location', 'target_end_date', 'Model'])
ls_all = ls_all.merge(ls_base, on=['location', 'target_end_date', 'Model', 'horizon', 'forecast_date'])
ls_all = ls_all.merge(ls_naive, on=['location', 'target_end_date', 'Model', 'horizon', 'forecast_date'])
ls_all = ls_all.merge(ls_hist, on=['location', 'target_end_date', 'Model', 'horizon', 'forecast_date'])
ls_all = ls_all.merge(ls_ets, on=['location', 'target_end_date', 'Model', 'horizon', 'forecast_date'])


ls_all['season'] = ls_all['forecast_date'].apply(lambda x: f'{str(x.year)}-{str(x.year+1)}' if \
                                                     x.week>=30 else f'{str(x.year-1)}-{str(x.year)}')



In [ ]:
ls_all.to_parquet('scores/ls_combined.parquet')